In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd

# Define the directory containing the images
folder_path = 'filepath'

# Supported image formats
supported_extensions = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')

# Define output list to collect data
data = []

# Define a variable to store the first image overlay for verification
first_overlay = None
first_filename = None

# Loop over all supported image files in the folder
for idx, filename in enumerate(sorted(os.listdir(folder_path))):
    if filename.lower().endswith(supported_extensions):
        image_path = os.path.join(folder_path, filename)

        # Load and preprocess image
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f"Warning: Unable to load {filename}. Skipping.")
            continue
        img_blur = cv2.GaussianBlur(img, (5, 5), 0)

        try:
            # Step 1: Create a clean specimen mask
            _, rough_mask = cv2.threshold(img_blur, 230, 255, cv2.THRESH_BINARY_INV)
            kernel = np.ones((5, 5), np.uint8)
            mask_closed = cv2.morphologyEx(rough_mask, cv2.MORPH_CLOSE, kernel)

            contours, _ = cv2.findContours(mask_closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if not contours:
                print(f"No specimen found in {filename}. Skipping.")
                continue
            largest = max(contours, key=cv2.contourArea)

            mask = np.zeros_like(img)
            cv2.drawContours(mask, [largest], -1, 255, thickness=cv2.FILLED)

            # Step 2: Find centroid
            M = cv2.moments(largest)
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])

            # Step 3: Calculate 20% area ROI
            specimen_area = np.sum(mask > 0)
            roi_area_target = 0.2 * specimen_area
            roi_radius = int(np.sqrt(roi_area_target / np.pi))

            yy, xx = np.ogrid[:img.shape[0], :img.shape[1]]
            circle_mask = ((xx - cx)**2 + (yy - cy)**2) <= roi_radius**2
            roi_mask = np.logical_and(circle_mask, mask > 0)
            surrounding_mask = np.logical_and(mask > 0, np.logical_not(roi_mask))

            # Step 4: Measurements
            full_specimen_intensity = np.mean(img_blur[mask > 0])
            center_roi_intensity = np.mean(img_blur[roi_mask])
            surrounding_specimen_intensity = np.mean(img_blur[surrounding_mask])
            intensity_ratio = center_roi_intensity / surrounding_specimen_intensity

            # Save data for this image
            data.append({
                'Filename': filename,
                'Specimen Area (pixels)': specimen_area,
                'Center ROI Target Area (pixels)': roi_area_target,
                'Calculated Center ROI Radius (pixels)': roi_radius,
                'Mean Intensity (Full Specimen)': full_specimen_intensity,
                'Mean Intensity (Center ROI)': center_roi_intensity,
                'Mean Intensity (Specimen Outside Center ROI)': surrounding_specimen_intensity,
                'Intensity Ratio (Center ROI / Specimen Outside Center ROI)': intensity_ratio
            })

            print(f"Processed: {filename}")

            # Save overlay for the very first valid image
            if first_overlay is None:
                overlay = cv2.cvtColor(img_blur.copy(), cv2.COLOR_GRAY2BGR)
                cv2.drawContours(overlay, [largest], -1, (0, 255, 0), 2)  # green outline
                cv2.circle(overlay, (cx, cy), roi_radius, (255, 0, 0), 2)  # blue center ROI
                cv2.circle(overlay, (cx, cy), 3, (0, 0, 255), -1)          # red centroid dot
                first_overlay = overlay
                first_filename = filename

        except Exception as e:
            print(f"Error processing {filename}: {e}")

# Create a DataFrame and save to CSV
df = pd.DataFrame(data)
output_csv_path = os.path.join(folder_path, 'compiled_intensity_data.csv')
df.to_csv(output_csv_path, index=False)

print("\nAll images processed.")
print(f"Results saved to: {output_csv_path}")

# Plot the first processed overlay for verification
if first_overlay is not None:
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(first_overlay, cv2.COLOR_BGR2RGB))
    plt.title(f"First Image Processed: {first_filename}\n(Green = Specimen, Blue = Center ROI, Red = Centroid)")
    plt.axis('off')
    plt.show()
else:
    print("No valid specimen images were processed, no verification image to show.")